# Create accessible tracks for paulette and rene waves, seperately
This code reads our AEW tracks from QTrack, grabs the paulette track or rene track (user specified), then saves it. 
- Paulette is saved as "second"
- Rene is saved as "main"

Important note: you will have to guess and check to find the wave identifier number ("no=11"). The waves corresponding to Rene are somewhere between 

In [56]:
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import pandas as pd
import os
import glob
from netCDF4 import Dataset

from AEW_module import season, AEW, AEW_CCKW

# for regridding
import xesmf as xe
import qtrack
import pickle
from qtrack.curvvort import compute_curvvort
from qtrack.tracking import run_postprocessing, run_tracking

from metpy.units import units


import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import wrf
from wrf import (to_np, interplevel, geo_bounds, getvar, smooth2d, get_cartopy, cartopy_xlim,
                 cartopy_ylim, latlon_coords, destagger)


In [57]:
## First determine if you want to track Rene or Paulette:
Paulette = True # set to false if you want to track Rene

In [58]:
## is this a restart run?
restart = False
# what is the base initialization time? (ens member)
init_time = pd.to_datetime('2020-09-03 03:00:00')
init_string = init_time.strftime('%d%H')
year = init_time.strftime('%Y')
# what time did you turn on the fluxes?
fluxon_time = pd.to_datetime('2020-09-05 12:00:00')
# this string is used to find that experiment
string = fluxon_time.strftime('%d%H')
# if this is not a restart run adjust the following
flux = 'fluxoff' # or 'fluxoff' 

In [59]:
if year == '2011':
    subdir = 'long_lived_case'
    if int(init_string) % 2 == 0:
        end_time = pd.to_datetime('2011-08-29 12:00') # Even 
    else:
        end_time = pd.to_datetime('2011-08-29 09:00') # Odd 
else:
    subdir = 'cent_atl_case'
    end_time = pd.to_datetime('2020-09-09 12:00')  
    
# for indexing...
date_list = pd.date_range(start=init_time, end=end_time, freq='3h')
fluxon_index = np.where(date_list==fluxon_time)[0][0]
response = fluxon_index + 3 # 3 days later, to respond to fluxes on

# Set directory where wrfout files reside, and list the files for processing.  Set up for a directory with only wrfout files.
if restart == True:
    plt_name = 'rst_on'+string+'z'
    f = pd.Timedelta(init_time - fluxon_time).total_seconds() 
    hours = int((f / 3600)*-1)
    run_name = 'rst_on'+str(hours)
else:
    run_name = flux
    plt_name = run_name
    hours = run_name

title = init_string+", "+run_name+", "+string+"z"
save_name = run_name +"_"+ init_string


In [60]:
ds = xr.open_dataset('AEW_tracks_post_proc_'+save_name+'.nc')


In [61]:
# This is the wave identifier. Might need to guess and check to find each wave
# Rene is usually somwhere between numbers 8-11
# Paulette is usually somewhere between 9-12
no = 9

In [62]:
lon = ds.sel(system=no).AEW_lon.values

In [63]:
lon

array([         nan,          nan,          nan,          nan,
                nan,          nan,          nan,          nan,
                nan,          nan,          nan,          nan,
                nan,          nan,          nan,          nan,
                nan,          nan,          nan,          nan,
                nan,          nan,          nan,          nan,
                nan,          nan,          nan,          nan,
                nan,  -3.55793963,  -5.67229435,  -7.04060135,
        -7.979476  ,  -9.09006183,  -9.79508063, -11.46767866,
       -13.85179645, -18.47742755, -24.48315403, -24.25924183,
       -24.27188173, -24.19321849, -24.11561671, -24.18517766,
       -24.42583229, -27.32890446, -30.46023726, -32.086059  ,
       -34.07287425, -36.25004651, -38.04545841, -39.51840734,
       -40.89932041, -41.68136777, -42.71630085, -44.75108103,
       -47.0434341 , -46.55390945, -48.49802763, -48.63898197,
       -48.77993631, -48.92089065, -49.82007587, -51.73

In [52]:
lat = ds.sel(system=no).AEW_lat.values

In [53]:
time = ds.sel(system=no).time.values

In [54]:
# Create one dataset with u, v winds at each lat, lon, time coordinate for this ensemble member
ds2 = xr.Dataset( 
    coords=dict(
        time=("time", time),
        lon=("lon", lon),
        lat=("lat", lat),
    ),
)

In [55]:
if Paulette == True:
    wave = 'second'   # Paulette
else: 
    wave = 'main'     # Rene

In [40]:
# apply regridding and save file
path_out = "/glade/u/home/athornton/qtrack/"+wave+"_wave/"
file_out = path_out +wave+'_wave_track_'+save_name+'.nc'
ds2.to_netcdf(path=file_out, format='NETCDF4', mode='w')